# CRISPR-BEAN + FUSE Workflow

End-to-end tutorial: from raw counts → Bayesian variant-effect inference → FUSE scores

---

## What is CRISPR-BEAN?

**BEAN** (Base Editing ANalysis) is a Bayesian framework for pooled CRISPR base-editing screens.
It accounts for per-guide editing efficiency (from the reporter) and chromatin accessibility to give calibrated variant-effect estimates.

### Two library designs

| | **Variant** | **Tiling** |
|---|---|---|
| gRNA design | Multiple gRNAs per target variant | Dense tiling across a locus |
| Bystander edits | Ignored | Fully modelled |
| Allele filtering step | Not needed | Required |
| Typical use | GWAS / non-coding variants | Coding-sequence saturation mutagenesis |

### Pipeline overview

```
raw FASTQs
    │
    ▼
bean count-samples   ──▶  ReporterScreen (.h5ad)
    │
    ▼
bean profile         ──▶  (optional) editing-pattern QC notebook
    │
    ▼
bean qc              ──▶  masked ReporterScreen (.h5ad)
    │
    ▼  [tiling only]
bean filter          ──▶  allele-filtered ReporterScreen (.h5ad)
    │
    ▼
bean run             ──▶  bean_element_result.*.csv
    │
    ▼  [coding tiling screens]
bean fuse            ──▶  bean_element_result.*.FUSE_score.csv
```


## 0. Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")

# Add repo to path (not needed if installed via pip)
REPO = "/sessions/dreamy-eloquent-noether/mnt/crispr-bean"
sys.path.insert(0, REPO)

import pandas as pd
import numpy as np
from plotnine import *

# ── Paths to test data bundled with the repo ──────────────────────────────────
DATA_DIR = f"{REPO}/tests/data"

VAR_H5AD        = f"{DATA_DIR}/var_mini_screen_annotated.h5ad"
TILING_H5AD     = f"{DATA_DIR}/tiling_mini_screen.h5ad"
VAR_RESULT_CSV  = f"{DATA_DIR}/var_res_example/bean_element_result.MixtureNormal+Acc.csv"
ACC_BW          = f"{DATA_DIR}/accessibility_signal_chr6.bw"

LDLR_EXCEL = f"{REPO}/examples/LDLR/0725_HCT116_LDLRPE_finalBEAN_FUSE_e.xlsx"
LDLR_DSS   = f"{REPO}/examples/LDLR/LDLR.dss"

print("Paths configured.")

---
## Step 1 · Count gRNA & reporter reads  (`bean count-samples`)

This step maps raw FASTQ files to gRNAs and counts editing outcomes in the reporter sequence.
It produces the core `ReporterScreen` object saved as `.h5ad`.

> **Note** – the test data was already counted; the command below shows the syntax for your own data.


In [ ]:
%%bash
# Example – run this on your own FASTQs
# bean count-samples \
#   --input /sessions/dreamy-eloquent-noether/mnt/crispr-bean/tests/data/sample_list.csv     \   # sample metadata + fastq paths
#   -b A                                   \   # edited base (A→G ABE; or C for CBE)
#   -f /sessions/dreamy-eloquent-noether/mnt/crispr-bean/tests/data/test_guide_info.csv      \   # gRNA metadata table
#   -o ./results/                          \   # output directory
#   -r                                     \   # quantify reporter edits
#   -n my_screen                               # screen identifier

echo "Count step would run here; using pre-counted test data instead."

### Required input files

**`sample_list.csv`** – one row per FASTQ file:

| Column | Description |
|---|---|
| `R1_filepath` | Path to R1 FASTQ |
| `R2_filepath` | Path to R2 FASTQ |
| `sample_id` | Unique sample label |
| `replicate` | Replicate number |
| `condition` | Sorting bin (`top`/`bot`/`bulk`) or time point (`D5`) |
| `upper_quantile` / `lower_quantile` | FACS sort gates (sorting screens) |

**`guide_info.csv`** – one row per gRNA:

| Column | Description |
|---|---|
| `name` | gRNA ID |
| `sequence` | 20-nt spacer |
| `target` | Target variant / element ID |
| `target_group` | `NegCtrl`, `PosCtrl`, or variant class |


---
## Step 2 · Inspect the `ReporterScreen` object

`ReporterScreen` extends AnnData.  Rows = gRNAs, columns = samples.


In [ ]:
import bean as be

# Load the pre-counted variant-screen test data
bdata = be.read_h5ad(VAR_H5AD)
print(bdata)

In [ ]:
# Guide (row) metadata
print("Guide columns:", bdata.guides.columns.tolist())
bdata.guides.head(5)

In [ ]:
# Sample (column) metadata
print("Sample columns:", bdata.samples.columns.tolist())
bdata.samples.head(8)

In [ ]:
# Count matrix:  rows = guides, cols = samples
import pandas as pd
X = pd.DataFrame(bdata.X, index=bdata.guides.index, columns=bdata.samples.index)
print("Count matrix shape:", X.shape)
X.head(5)

In [ ]:
# Guide-level summary statistics
guide_totals = pd.DataFrame({
    "total_counts": bdata.X.sum(axis=1),
    "target":       bdata.guides["target"].values,
    "target_group": bdata.guides["target_group"].values,
})
print("Guides by target_group:")
print(guide_totals["target_group"].value_counts())

In [ ]:
# Distribution of per-guide read depth
df_counts = pd.DataFrame({
    "log10_counts": np.log10(guide_totals["total_counts"].clip(lower=1)),
    "target_group": guide_totals["target_group"],
})

(
    ggplot(df_counts, aes(x="log10_counts", fill="target_group"))
    + geom_histogram(bins=40, alpha=0.7, position="identity")
    + labs(title="Per-guide total read depth", x="log10(total counts)", y="# guides")
    + theme_bw()
    + scale_fill_brewer(type="qual", palette="Set2")
)

---
## Step 3 · Quality control  (`bean qc`)

QC checks guide coverage, count correlations between replicates, and editing rates.
Low-quality samples are masked in `bdata.samples["mask"]`.


In [ ]:
%%bash
# bean qc \
#   /sessions/dreamy-eloquent-noether/mnt/crispr-bean/tests/data/var_mini_screen.h5ad     \   # input
#   -o /sessions/dreamy-eloquent-noether/mnt/crispr-bean/tests/data/var_mini_screen_masked.h5ad \
#   -r ./qc_report_var_mini                   \
#   --control-condition bulk

echo "QC step – inspect bdata.samples for mask column after running."

In [ ]:
# After QC, samples that fail have mask=True.
# The masked .h5ad is then used downstream.

# Demonstrate what masking looks like on the test data
if "mask" in bdata.samples.columns:
    n_masked = bdata.samples["mask"].sum()
    print(f"Masked samples: {n_masked} / {len(bdata.samples)}")
    print(bdata.samples[["condition", "replicate", "mask"]].head(10))
else:
    print("No mask column – all samples pass QC in this test dataset.")

In [ ]:
# Replicate correlation (proxy for what bean qc checks)
conds = bdata.samples["condition"].unique()
print("Conditions in dataset:", conds)

# Log-fold change between first two conditions
c1, c2 = conds[0], conds[1]
s1 = bdata.samples["condition"] == c1
s2 = bdata.samples["condition"] == c2
lfc = np.log2(
    (bdata.X[:, s2].mean(axis=1) + 1) /
    (bdata.X[:, s1].mean(axis=1) + 1)
)

df_lfc = pd.DataFrame({
    "lfc":          lfc,
    "target_group": bdata.guides["target_group"].values,
})

(
    ggplot(df_lfc, aes(x="target_group", y="lfc", fill="target_group"))
    + geom_boxplot(alpha=0.7, outlier_alpha=0.2)
    + labs(
        title=f"Log-fold change ({c1} → {c2}) by guide class",
        x="", y="LFC"
    )
    + coord_flip()
    + theme_bw()
    + theme(legend_position="none")
    + scale_fill_brewer(type="qual", palette="Set2")
)

---
## Step 4 · Filter alleles  (`bean filter`) — tiling screens only

For tiling libraries, every observed editing pattern is a candidate variant.
This step:
1. Keeps only edits within the intended base-change type (A→G or C→T)
2. Filters by editing window position in the spacer
3. Removes rare alleles below a count/proportion threshold
4. (Optional) Translates nucleotide edits → amino acid changes for coding targets


In [ ]:
%%bash
# Example for a tiling screen targeting a coding sequence
# bean filter /sessions/dreamy-eloquent-noether/mnt/crispr-bean/tests/data/tiling_mini_screen_masked.h5ad \
#   -o /sessions/dreamy-eloquent-noether/mnt/crispr-bean/tests/data/tiling_alleleFiltered \
#   --filter-target-basechange \
#   --filter-window --edit-start-pos 0 --edit-end-pos 19 \
#   --filter-allele-proportion 0.1 \
#   --filter-sample-proportion 0.3 \
#   --translate \
#   --translate-gene-name LDLR

echo "Filter step demonstrated via the pre-processed tiling data below."

In [ ]:
# Inspect the tiling screen's allele counts
bdata_tiling = be.read_h5ad(TILING_H5AD)
print(bdata_tiling)
print()
print("Guide columns:", bdata_tiling.guides.columns.tolist())

In [ ]:
# Allele counts are stored in .uns
if "allele_counts" in bdata_tiling.uns:
    ac = bdata_tiling.uns["allele_counts"]
    if isinstance(ac, dict):
        key = list(ac.keys())[0]
        print(f"Allele table key: {key!r}")
        df_ac = ac[key]
    else:
        df_ac = ac
    print(f"Allele count table shape: {df_ac.shape}")
    print(df_ac.head(5).to_string())
else:
    print("Allele counts not present in this test object.")

---
## Step 5 · Bayesian inference  (`bean run`)

`bean run` fits a Pyro probabilistic model to infer the effect size (`mu`) of each variant.
It outputs per-variant effect sizes and calibrated z-scores.

### Model overview

For each variant *v*, the model estimates:

- **`mu`** – mean phenotypic effect (log-scale)
- **`mu_sd`** – posterior standard deviation of the effect
- **`mu_z`** – raw z-score = `mu / mu_sd`

When negative controls are available, two calibration stages adjust the z-score:

| Stage | Output | What it does |
|---|---|---|
| 1A (`--fit-negctrl`) | `mu_z_scaled` | Centers by negctrl mean, scales by negctrl SD |
| 1B (auto if ≥10 negctrls) | `mu_z_adj` | Re-scales `mu_sd` so negctrl z-scores have SD≈1 |

**Use `mu_z_adj` when available.** Without negative controls only `mu_z` is produced, which assumes N(0,1) null but does not verify it.


In [ ]:
%%bash
# Run on the variant test screen (fast: --n-iter 200 for demo)
# bean run sorting variant \
#   /sessions/dreamy-eloquent-noether/mnt/crispr-bean/tests/data/var_mini_screen_annotated.h5ad \
#   -o /tmp/bean_run_out/ \
#   --scale-by-acc --acc-bw-path /sessions/dreamy-eloquent-noether/mnt/crispr-bean/tests/data/accessibility_signal_chr6.bw \
#   --fit-negctrl \
#   --n-iter 200

echo "Using pre-computed results for the demo."

In [ ]:
# Load the pre-computed variant-screen result
df_res = pd.read_csv(VAR_RESULT_CSV, index_col=0)
print("Result shape:", df_res.shape)
print("Columns:", df_res.columns.tolist())

In [ ]:
df_res.head(8)

### 5.1 · Z-score distributions

The best available z-score is **`mu_z_adj`** > `mu_z_scaled` > `mu_z`.


In [ ]:
# Which z-score columns are present?
z_cols = [c for c in ["mu_z", "mu_z_scaled", "mu_z_adj"] if c in df_res.columns]
print("Available z-score columns:", z_cols)
best_z = z_cols[-1]
print(f"Using: '{best_z}'")

In [ ]:
# Distribution of best z-score
(
    ggplot(df_res.dropna(subset=[best_z]), aes(x=best_z))
    + geom_histogram(bins=50, fill="#4878CF", color="white", alpha=0.8)
    + geom_vline(xintercept=[-2, 2], linetype="dashed", color="red", alpha=0.6)
    + labs(title=f"Distribution of {best_z}", x=best_z, y="# variants")
    + theme_bw()
)

### 5.2 · Effect size vs uncertainty  (`mu` vs `mu_sd`)

`mu_sd` reflects how confident the model is in the effect estimate.
A `--mu-sd-max` filter (used in FUSE) keeps only well-estimated variants.


In [ ]:
df_plot = df_res.dropna(subset=["mu", "mu_sd", best_z]).copy()

# Colour by |z| > 2 threshold
df_plot["significant"] = df_plot[best_z].abs() > 2

(
    ggplot(df_plot, aes(x="mu", y="mu_sd", color="significant"))
    + geom_point(alpha=0.5, size=1.5)
    + scale_color_manual(values={True: "#e74c3c", False: "#95a5a6"})
    + labs(
        title="Effect size vs posterior uncertainty",
        x="mu (effect size)", y="mu_sd (uncertainty)",
        color="|z| > 2",
    )
    + theme_bw()
)

### 5.3 · Null calibration check

When negative controls are present, the calibrated z-scores should approximately
follow N(0,1) for the null variants.  Use this as a sanity check.


In [ ]:
from scipy.stats import norm as scipy_norm

df_z = df_res[[best_z]].dropna().copy()
df_z.columns = ["z"]

# Overlay theoretical N(0,1)
x_grid = np.linspace(df_z["z"].quantile(0.001), df_z["z"].quantile(0.999), 200)
df_theory = pd.DataFrame({"z": x_grid,
    "density": scipy_norm.pdf(x_grid, 0, 1) * len(df_z) * (df_z["z"].max()-df_z["z"].min())/50})

(
    ggplot(df_z, aes(x="z"))
    + geom_histogram(bins=50, fill="#4878CF", alpha=0.7, color="white")
    + geom_line(data=df_theory, mapping=aes(x="z", y="density"), color="red", size=1)
    + labs(
        title=best_z + " distribution vs N(0,1) (red)",
        x=best_z, y="count",
    )
    + theme_bw()
)

### 5.4 · Top hits

Variants with the most extreme z-scores.


In [ ]:
df_hits = (
    df_res[[best_z, "mu", "mu_sd", "target", "target_group"]]
    .dropna(subset=[best_z])
    .sort_values(best_z)
)

print("Top negative effect variants (most depleted):")
print(df_hits.head(10).to_string(index=False))
print()
print("Top positive effect variants (most enriched):")
print(df_hits.tail(10).to_string(index=False))

### 5.5 · Group-level comparison

`mu_z` (or `mu_z_adj`) can be used for group-level tests.
For example: are missense variants different from synonymous variants?

> This is statistically valid even without negctrl calibration — you're comparing
> two distributions, not relying on the absolute N(0,1) null.


In [ ]:
# Use target_group as a proxy for variant class in this test dataset
groups = df_res.groupby("target_group")[best_z].apply(list).to_dict()

print("Variant groups and counts:")
for g, vals in groups.items():
    print(f"  {g:20s}  n={len(vals):4d}  median z={np.nanmedian(vals):.3f}")

# Plot
df_group = df_res[["target_group", best_z]].dropna()
(
    ggplot(df_group, aes(x="target_group", y=best_z, fill="target_group"))
    + geom_boxplot(alpha=0.7, outlier_alpha=0.15, outlier_size=1)
    + coord_flip()
    + labs(title=best_z + " by variant group", x="", y=best_z)
    + theme_bw()
    + theme(legend_position="none")
    + scale_fill_brewer(type="qual", palette="Set2")
)

---
## Step 6 · FUSE scores  (`bean fuse`)

**FUSE** (Functional Score Using Structural Ensemble) denoises per-variant
functional scores by combining:

- **Positional component** – James-Stein shrinkage estimate of the mean effect
  across all observed substitutions at the same amino-acid position
- **Substitution component** – amino-acid pair score from the FUNSUM matrix
  (a substitution matrix trained on DMS datasets)

$$\text{FUSE score} = \text{pos\_score} + \text{sub\_score}$$

When a DSSP secondary-structure file is supplied, the substitution component
uses a secondary-structure-specific FUNSUM (`FUSE_SS_score`).

### When to use FUSE
- Coding tiling screens producing per amino-acid substitution scores
- When you want to impute scores for substitutions not directly observed
- When you want to leverage structural information for denoising


### 6.1 · What the FUSE pipeline expects

The input is a **bean element result CSV** with amino-acid edit strings in one column.
The edit string format is `[gene:]pos:ref>alt`, e.g. `LDLR:646:L>*` or `646:L>*`.

| Flag | Default | Meaning |
|---|---|---|
| `--edit-col` | `target` | Column with edit strings |
| `--score-col` | auto | `mu_z_adj` if present, else `mu_z` |
| `--mu-sd-max` | off | Filter variants with `mu_sd > threshold` |
| `--include-lof` | on | Include stop-gain variants |
| `--exclude-lof` | — | Use noLOF FUNSUM, drop stop-gains |
| `--dss` | off | DSSP file for SS-specific FUNSUM |


In [ ]:
%%bash
# Run bean fuse from the command line
# bean fuse bean_element_result.MixtureNormal.csv \
#   --edit-col target \
#   --mu-sd-max 1.0 \
#   --dss /path/to/protein.dss \
#   -o results/

echo "Demonstrating the Python API below."

### 6.2 · Build a BEAN-format input from the LDLR peDMS data

We use the real LDLR prime-editing DMS data bundled in `fuse_temp/` as a
demonstration (normally this would be the output of `bean run`).


In [ ]:
# ── Build a minimal bean element result CSV ───────────────────────────────────
import sys
sys.path.insert(0, REPO)

df_ldlr = pd.read_excel(LDLR_EXCEL, sheet_name="data")

df_bean_input = pd.DataFrame({
    # Edit string in bean format: "pos:ref>alt"
    "target": (
        df_ldlr["LDLR pos"].astype(str).str.strip() + ":" +
        df_ldlr["Starting AA"].astype(str).str.strip() + ">" +
        df_ldlr["Final AA"].astype(str).str.strip().str.replace("Z", "*")
    ),
    "mu_z_adj": pd.to_numeric(df_ldlr["mu_z_adj"],   errors="coerce"),
    "mu_sd":    pd.to_numeric(df_ldlr["mu_sd"],       errors="coerce"),
})

# Keep only missense / stop-gain / synonymous
keep = (
    df_ldlr["Missense?"].fillna(False) |
    df_ldlr["Stop?"].fillna(False)     |
    df_ldlr["Synonymous?"].fillna(False)
)
df_bean_input = df_bean_input[keep].dropna(subset=["mu_z_adj"]).reset_index(drop=True)

print(f"Input rows: {len(df_bean_input)}")
df_bean_input.head(6)

In [ ]:
# Save to a temporary CSV
import tempfile, os
tmp_csv = os.path.join(tempfile.gettempdir(), "ldlr_bean_result.csv")
df_bean_input.to_csv(tmp_csv)
print(f"Saved to {tmp_csv}")

### 6.3 · Compute FUSE scores — without secondary structure

In [ ]:
import importlib.util

def _load(name, path):
    if name in sys.modules:
        del sys.modules[name]
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

_base = f"{REPO}/bean/fuse"
for _n, _p in [
    ("bean.fuse.funsum", f"{_base}/funsum.py"),
    ("bean.fuse.dssp",   f"{_base}/dssp.py"),
    ("bean.fuse.utils",  f"{_base}/utils.py"),
    ("bean.fuse.score",  f"{_base}/score.py"),
]:
    _load(_n, _p)

from bean.fuse.score import calculate_fuse_scores

df_fuse_no_ss = calculate_fuse_scores(
    input_csv   = tmp_csv,
    edit_col    = "target",
    include_lof = True,
    dss_path    = None,          # no secondary structure
    gene_name   = "LDLR",
    mu_sd_max   = 1.0,           # filter high-uncertainty variants
)
print(f"FUSE output: {len(df_fuse_no_ss)} rows (all possible substitutions at observed positions)")
df_fuse_no_ss.head(8)

### 6.4 · Compute FUSE scores — with DSSP secondary structure

In [ ]:
df_fuse = calculate_fuse_scores(
    input_csv   = tmp_csv,
    edit_col    = "target",
    include_lof = True,
    dss_path    = LDLR_DSS,      # DSSP file from PDB structure
    gene_name   = "LDLR",
    mu_sd_max   = 1.0,
)
print(f"FUSE output with SS: {len(df_fuse)} rows")
df_fuse[["gene","aapos","aaref","aaalt","functional_class",
         "ss","raw_score","norm_raw_score",
         "pos_score","sub_score","sub_score_ss",
         "FUSE_score","FUSE_SS_score"]].head(8)

### 6.5 · FUSE score distributions by functional class

In [ ]:
# Keep one row per observed variant (those with a raw_score)
df_obs = df_fuse.dropna(subset=["raw_score"]).copy()

(
    ggplot(df_obs, aes(x="FUSE_SS_score", fill="functional_class"))
    + geom_density(alpha=0.5)
    + scale_fill_manual(values={"SYN": "#2ecc71", "MIS": "#3498db", "LOF": "#e74c3c"})
    + labs(
        title="FUSE_SS_score distribution by functional class",
        x="FUSE_SS_score", y="Density", fill="Class",
    )
    + theme_bw()
)

### 6.6 · Raw score vs FUSE score — denoising effect

FUSE smooths noisy raw scores by borrowing strength from:
- nearby positions (positional component)
- the FUNSUM substitution matrix (substitution component)


In [ ]:
df_cmp = df_fuse.dropna(subset=["raw_score"]).copy()

(
    ggplot(df_cmp, aes(x="raw_score", y="FUSE_SS_score", color="functional_class"))
    + geom_point(alpha=0.4, size=1.2)
    + geom_abline(slope=1, intercept=0, linetype="dashed", color="grey")
    + scale_color_manual(values={"SYN": "#2ecc71", "MIS": "#3498db", "LOF": "#e74c3c"})
    + labs(
        title="Raw score vs FUSE_SS_score",
        x="raw score (mu_z_adj, normalized)",
        y="FUSE_SS_score",
        color="Class",
    )
    + theme_bw()
)

### 6.7 · Positional map of FUSE scores

Visualise mean FUSE score along the protein to spot functional hotspots.


In [ ]:
df_pos_avg = (
    df_fuse[df_fuse["functional_class"] == "MIS"]
    .groupby("aapos", as_index=False)["FUSE_SS_score"]
    .mean()
    .rename(columns={"FUSE_SS_score": "mean_FUSE"})
)

(
    ggplot(df_pos_avg, aes(x="aapos", y="mean_FUSE"))
    + geom_line(color="#3498db", size=0.4, alpha=0.7)
    + geom_smooth(method="loess", color="#e74c3c", se=False, size=1.2, span=0.1)
    + labs(
        title="Mean FUSE score per position (missense only)",
        x="Amino-acid position", y="Mean FUSE_SS_score",
    )
    + theme_bw()
)

### 6.8 · Secondary-structure-specific differences

Positions in helices, strands, and loops can have different FUNSUM priors.


In [ ]:
# Map DSSP codes to grouped labels
SS_GROUP = {
    "G": "Helix", "H": "Helix", "I": "Helix",
    "E": "Strand", "B": "Strand",
    "T": "Loop",   "S": "Loop",   "C": "Loop",
}
df_fuse["ss_group"] = df_fuse["ss"].map(SS_GROUP)
df_ss = df_fuse.dropna(subset=["ss_group", "raw_score"]).copy()

(
    ggplot(df_ss, aes(x="ss_group", y="FUSE_SS_score", fill="ss_group"))
    + geom_boxplot(alpha=0.7, outlier_alpha=0.2, outlier_size=1)
    + scale_fill_manual(values={"Helix": "#e74c3c", "Strand": "#3498db", "Loop": "#2ecc71"})
    + labs(title="FUSE_SS_score by secondary structure", x="", y="FUSE_SS_score")
    + theme_bw()
    + theme(legend_position="none")
)

### 6.9 · Validate against pre-computed reference scores

The `fuse_temp/functional_datasets/` folder contains pre-computed FUSE scores
calculated using the original R implementation.  The Pearson correlation below
confirms that the Python port reproduces the reference results.


In [ ]:
# Load reference FUSE scores from the Excel file
pre = df_ldlr[["Starting AA","LDLR pos","Final AA","FUSE_score","FUSE_SS_score"]].dropna(subset=["FUSE_score"]).copy()
pre["gene_aa_str"] = (
    "LDLR---" +
    pre["Starting AA"].str.strip() +
    pre["LDLR pos"].astype(int).astype(str) +
    pre["Final AA"].str.strip().str.replace("Z","*")
)
merged = df_fuse.merge(pre[["gene_aa_str","FUSE_score","FUSE_SS_score"]],
                       on="gene_aa_str", suffixes=("_python","_r"))

r1 = merged[["FUSE_score_python","FUSE_score_r"]].corr().iloc[0,1]
r2 = merged[["FUSE_SS_score_python","FUSE_SS_score_r"]].corr().iloc[0,1]

print(f"Correlation Python vs R  (n={len(merged)} variants):")
print(f"  FUSE_score     Pearson r = {r1:.4f}")
print(f"  FUSE_SS_score  Pearson r = {r2:.4f}")

(
    ggplot(merged, aes(x="FUSE_score_r", y="FUSE_score_python", color="FUSE_score_r"))
    + geom_point(alpha=0.3, size=1)
    + geom_abline(slope=1, intercept=0, linetype="dashed", color="red")
    + scale_color_gradient(low="#3498db", high="#e74c3c")
    + labs(
        title="Python vs R FUSE scores  (r = " + f"{r1:.3f})",
        x="FUSE_score (R reference)",
        y="FUSE_score (Python)",
    )
    + theme_bw()
    + theme(legend_position="none")
)

---
## Summary

| Step | Command | Key output |
|---|---|---|
| Count | `bean count-samples` | `screen.h5ad` |
| (Profile) | `bean profile` | editing-pattern QC |
| QC | `bean qc` | `screen_masked.h5ad`, QC report |
| Filter *(tiling)* | `bean filter` | `screen_alleleFiltered.h5ad` |
| Infer effects | `bean run` | `bean_element_result.*.csv` |
| FUSE *(coding)* | `bean fuse` | `*.FUSE_score.csv` |

### Key z-score concepts

- **`mu_z`** – raw Bayesian posterior z-score; assumes N(0,1) null (unverified without negctrls)
- **`mu_z_scaled`** – Stage 1A: anchored to negctrl mean/SD
- **`mu_z_adj`** – Stage 1B: re-calibrated so negctrl z-scores have empirical SD ≈ 1 → proper N(0,1) null
- For group-level comparisons (e.g. missense vs synonymous), `mu_z` is fine regardless of calibration

### FUSE output columns

| Column | Meaning |
|---|---|
| `raw_score` | Input `mu_z_adj` (or `mu_z`) for observed variants |
| `norm_raw_score` | Normalised to SYN≈0, LOF≈1 scale |
| `pos_score` | James-Stein positional mean |
| `sub_score` | Overall FUNSUM substitution score |
| `sub_score_ss` | Secondary-structure-specific FUNSUM score |
| `FUSE_score` | `pos_score + sub_score` |
| `FUSE_SS_score` | `pos_score + sub_score_ss` (preferred when DSS available) |
